# Chi-Squared Selector

A maldição das features:
- Nem sempre ter mais features significa um modelo mais competente
- É necessário identificar quais são as features que mais contribuem para o sucesso de um modelo
- chi squared para seleção de atributos: técnica que busca melhorar o modelo pela seleção de um subconjunto de features mais relevantes para prever a classe
- Usa o teste de independência do chi-squared para selecionar os atributos
- Recebe uma coluna com um vetor de atributos e produz uma coluna com um vetor de atributos mais relevantes

Parâmetros:
- selectorType: a forma de selecionar os atributos
- numTopFeatures: quantos atributos devem ser selecionados
- percentile: percentual de atributos que devem ser selecionados
- fpr e fwe: seleciona atributos que os p-values estejam abaixo de um parâmetro. Padrão 0.05
- fdr: aplica o critério de FDR (False Discovery Rate). Padrão 0.05

In [0]:
pip install findspark

In [0]:
import findspark, pyspark
from pyspark.sql import SparkSession
findspark.init()
spark = SparkSession.builder.appName('chi_squared_selector').getOrCreate()

In [0]:
from pyspark.ml.feature import ChiSqSelector, RFormula

In [0]:
abs_path = "/Volumes/workspace/ml_with_spark/raw_data/Carros.csv"

carros = spark.read.csv(abs_path, header=True, inferSchema=True, sep=';')
carros.show(5)

In [0]:
rformula = RFormula(formula='HP ~ .', featuresCol='independente', labelCol='dependente')
carros_rf = rformula.fit(carros).transform(carros)
carros_rf.select('independente', 'dependente').show(5, truncate=False)

In [0]:
selector = ChiSqSelector(
    selectorType='fdr', 
    fdr=0.01, 
    featuresCol='independente', 
    outputCol='selecionados',
    labelCol='dependente'
)
carros_cqs = selector.fit(carros_rf).transform(carros_rf)
carros_cqs.select('selecionados').show(5, truncate=False)